In [2]:
import json
from pathlib import Path

input_path = Path(
    r"C:\Users\JesperStåhl\dbt_projects\demo_project\target\run_results.json"
)

with input_path.open() as f:
    run_results = json.load(f)

print("Keys in file:", run_results.keys())
print("Number of executed items:", len(run_results["results"]))


Keys in file: dict_keys(['metadata', 'results', 'elapsed_time', 'args'])
Number of executed items: 16


In [5]:
tests = [
    r for r in run_results["results"]
    if r["unique_id"].startswith("test.")
]

print(f"Number of tests: {len(tests)}")
print("Example test ID:", tests[0]["unique_id"])

Number of tests: 16
Example test ID: test.demo_project.no_invalid_order_amounts_fact_orders_order_amount.2aeb76a2fe


In [6]:
summary = []

for test in tests:
    summary.append({
        "name": test["unique_id"].split(".")[-1],
        "status": test["status"],
        "failures": test.get("failures", 0),
        "message": test.get("message"),
        "execution_time_sec": round(test["execution_time"], 3)
    })

summary[:3]


[{'name': '2aeb76a2fe',
  'status': 'pass',
  'failures': 0,
  'message': None,
  'execution_time_sec': 0.099},
 {'name': '91beb8c926',
  'status': 'pass',
  'failures': 0,
  'message': None,
  'execution_time_sec': 0.113},
 {'name': '736d86e794',
  'status': 'pass',
  'failures': 0,
  'message': None,
  'execution_time_sec': 0.093}]

In [7]:
total = len(summary)
passed = sum(1 for t in summary if t["status"] == "pass")
failed = sum(1 for t in summary if t["status"] == "fail")
warned = sum(1 for t in summary if t["status"] == "warn")
errored = sum(1 for t in summary if t["status"] == "error")

total, passed, failed, warned, errored

(16, 15, 1, 0, 0)

In [8]:
lines = []
lines.append("DBT TEST RUN SUMMARY")
lines.append("====================")
lines.append("")
lines.append(f"Total tests: {total}")
lines.append(f"Passed: {passed}")
lines.append(f"Failed: {failed}")
lines.append(f"Warnings: {warned}")
lines.append(f"Errors: {errored}")
lines.append("")

if failed or errored:
    lines.append("FAILED TESTS")
    lines.append("------------")
    for t in summary:
        if t["status"] in ("fail", "error"):
            lines.append(
                f"- {t['name']} | failures={t['failures']} | message={t['message']}"
            )

print("\n".join(lines))

DBT TEST RUN SUMMARY

Total tests: 16
Passed: 15
Failed: 1
Warnings: 0
Errors: 0

FAILED TESTS
------------
- 5fb22c2710 | failures=1 | message=Got 1 result, configured to fail if != 0


In [9]:
output_path = Path(
    r"C:\Users\JesperStåhl\dbt_projects\demo_project\testrun_summary.txt"
)

with output_path.open("w", encoding="utf-8") as f:
    f.write("\n".join(lines))

print(f"Summary written to: {output_path}")

Summary written to: C:\Users\JesperStåhl\dbt_projects\demo_project\testrun_summary.txt
